<a href="https://colab.research.google.com/github/joshSi/socbench/blob/carbanak/PRISM26_CARBANAKv2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PRISM 2026: CARBANAKv2 Tutorial

**Note:** Please **File → Save a copy in Drive** before running this notebook, and do all work in **your own copy**.


## Step 0: Install Tutorial Assets

*Feel free to run this as soon as you save a copy of this notebook to your Google Drive!*


In [ ]:
!apt-get -qq update
!apt-get -qq install -y libcairo2-dev pkg-config python3-dev
!pip -q install python-igraph cairocffi tqdm matplotlib

import csv
import igraph
import json
import os
import re
import shlex
import subprocess


def fetch(url, out_path):
    os.makedirs(os.path.dirname(out_path) or ".", exist_ok=True)

    # -L follow redirects, --fail makes non-200 an error, --retry helps transient failures
    cmd = f'curl -L --fail --retry 5 --retry-delay 2 -o {shlex.quote(out_path)} {shlex.quote(url)}'
    print("GET:", out_path)
    r = subprocess.run(cmd, shell=True)
    if r.returncode != 0:
        raise RuntimeError(f"Download failed for {out_path}")

    size = os.path.getsize(out_path)
    if size == 0:
        raise RuntimeError(f"Downloaded 0 bytes for {out_path} (likely auth/redirect issue)")

    print(f"  OK ({size/1024/1024:.2f} MB)")

%cd /content
!rm -rf prism_assets
!mkdir -p prism_assets/graphs
%cd prism_assets

FILES = {
    # Graph Analysis functions from the REAPr analysis scripts
    "graph_helpers.py":          "https://bitbucket.org/sts-lab/reapr-ground-truth/raw/main/utilities/graph_helpers.py",
    # Data files that faciliate Liu, Inam, et al.'s PRISM'26
    #   provenance tracing heuristics for EDR telemetry
    "empirical_filter_set.json": "https://bitbucket.org/sts-lab/reapr-ground-truth/raw/main/utilities/empirical_filter_set.json",
    "path_cluster_map.pkl.xz":   "https://bitbucket.org/sts-lab/reapr-ground-truth/raw/main/utilities/path_cluster_map.pkl.xz",
    "cluster_rep_map.pkl.xz":    "https://bitbucket.org/sts-lab/reapr-ground-truth/raw/main/utilities/cluster_rep_map.pkl.xz",
}

# Attack graphs generated from *time-sliced* copies of the log.
# Still lots and lots of benign activity that needs denoising.
GRAPH_TARS = {
    "graphs/h1_graph_sliced.pkl.tar.gz": "https://bitbucket.org/sts-lab/carbanakv2/raw/main/attack_graphs/h1_graph_sliced.pkl.tar.gz",
    "graphs/fs_graph_sliced.pkl.tar.gz": "https://bitbucket.org/sts-lab/carbanakv2/raw/main/attack_graphs/fs_graph_sliced.pkl.tar.gz",
}

# Download utility files
for dst, url in FILES.items():
    fetch(url, dst)

# Download + extract graphs
for dst, url in GRAPH_TARS.items():
    fetch(url, dst)

    # Validate archive + extract (quiet)
    !tar -tzf "{dst}" > /dev/null 2>&1
    !tar -xzf "{dst}" -C graphs > /dev/null 2>&1

print("\nFinal contents:\n")
!ls -lh
!ls -lh graphs



## Step 1: Load an Attack Graph

There are two different attack graphs to play with:

*   H1 is a Windows device that is the initial point of access.
*   FS is a Linux fileserver. The attack spreads to here next.

Let's play with H1 first.

The graph_helpers library imported here is from [REAPr attack labeling code base](https://bitbucket.org/sts-lab/reapr-ground-truth/src/master/utilities/).

In [ ]:
from graph_helpers import pickle_read

# Windows Device
igraph_file = "graphs/h1_graph_sliced.pickle"

# Linux Device
# igraph_file = "graphs/fs_graph_sliced.pickle"

g = pickle_read(igraph_file)


print("Loaded:", igraph_file)
print("V =", len(g.vs), "E =", len(g.es),"\n")

print("Example Output:")
for v in g.vs[:10]:
    print(v.index, v["type"], v["name"], v["uuid"])

## Step 2: Search the Graph for IoC's

The qualitative log of attack steps is available on [Google Drive](https://docs.google.com/spreadsheets/d/1RAUmYx5YcvPGcTofX-whONUbaKiqefZSPTpVnR2DIik/edit?usp=sharing) and the [git repository](https://bitbucket.org/sts-lab/carbanakv2/src/main/docs/attack_log.csv). We'll be using this as a shortcut to find indicators of compromise (IoC's).

In order for our analysis to work, **we need to identify at least one root cause and one impact IOC.** Root causes are typically an entry point for the attacker onto the device. Impacts are typically an exit point from the device, a data artifact written by the attacker, a process they left running, etc.

### Search Graph Vertices by Name/Label

To search the graph, update the `QUERY` variable in the code below

Some useful artifacts to get you started:

For the H1 attack graph...

*   The attacker's IP address is `10.195.179.70` and connects by reverse shell over port `8080`.
*   The attacker eventually SSH's from H1 to FS's IP address is `130.126.136.248` over Port 22.

For the FS attack graph...

*   H1's IP address is `10.195.75.103`
*   The Domain Controller's IP address is `130.126.136.46`



In [ ]:
# Edit this:
QUERY = "EDIT ME"

def search_vertices(g, pattern, max_results=200):
    rx = re.compile(pattern, re.IGNORECASE)

    hits = []
    for v in g.vs:
        name = v["name"] if "name" in v.attributes() else ""
        if name is None:
            name = ""

        if rx.search(str(name)):
            hits.append(v.index)
            if len(hits) >= max_results:
                break

    print(f"Regex: {pattern}")
    print(f"Matches: {len(hits)} (showing up to {max_results})\n")

    for vid in hits:
        v = g.vs[vid]
        print(f"v{vid}")
        print(f"  type: {v['type']}")
        print(f"  name: {v['name']}")
        print(f"  uuid: {v['uuid']}")
        print()

    return hits

hits = search_vertices(g, QUERY, max_results=200)

## Step 3: Create a Graph seeds file

In a minute, we'll trace the attack graph using the [REAPr Framework](https://bitbucket.org/sts-lab/reapr-ground-truth/). In a nutshell, these scripts will conduct a forward trace from root causes, conduct a backward trace from the impacts, and then leave us with the intersection of the two.

For the sake of reproducibility, we'll write our seed labels our to a file, then have that be read in by the next code block.

In [ ]:
# Edit this:
root_cause_uuids = [
    # "uuid-here",
]

# Edit this:
impact_uuids = [
    # "uuid-here",
   ]

# Write: vid, type, name, uuid, impact/root_cause
# Output: graphs/{hostname}_graph_sliced.seeds
def write_seeds_file(g, igraph_file, root_uuids, impact_uuids):
    assert igraph_file.endswith(".pickle"), "Expected .pickle igraph file"
    out_path = igraph_file.replace(".pickle", ".seeds")

    wrote = 0
    with open(out_path, "w", newline="") as f:
        w = csv.writer(f)

        for v in g.vs:
            uuid = v["uuid"]

            if uuid in root_uuids:
                label = "root_cause"
            elif uuid in impact_uuids:
                label = "impact"
            else:
                continue

            w.writerow([v.index, v["type"], v["name"], uuid, label])
            wrote += 1

    print("Wrote seeds file:", out_path)
    print("Rows written:", wrote)
    return out_path

seed_file = write_seeds_file(g, igraph_file, root_cause_uuids, impact_uuids)

!echo "=== seeds preview ==="
!head -n 25 "$seed_file"



## Step 4: Run REAPr Analysis

This code is copied over from [reapr.py](https://bitbucket.org/sts-lab/reapr-ground-truth/src/master/utilities/reapr.py) so that you can review the logic.

Key Steps:
1.   `color_graph` performs the forward/back trace intersection.
2.   `attack_only` prunes all vertices that were not marked in the traces.
3.   the `process_mode` and `data_mode` labels correspond to the Windows tracing heuristics presented in Liu, Inam, et al.'s PRISM'26 paper.
4.   Clean up redundant edges with `prune_edges`.
5.   Clean up orphaned vertices with `remove_orphans`.

A full set of labels has also been written to a CSV file.

In [ ]:
from graph_helpers import (
    pickle_read,
    relabels,
    vertex_times,
    color_graph,
    attack_only,
    remove_singleton_data,
    prune_vertices,
    contract_clustered_files,
    prune_edges,
    remove_orphan_data,
    relabel_name,
    dump_edges,
    dump_vertices,
    write_labels_file_attack_only,
)

process_mode = "all"   # inter | intra | all | none
data_mode    = "all"   # cluster | leaf | all | none

base_fname = igraph_file.replace(".pickle", "")
seed_file  = base_fname + ".seeds"
pdf_file   = base_fname + ".pdf"
png_file = base_fname + ".png"

print("igraph:", igraph_file)
print("seeds:", seed_file)
print("pdf  :", pdf_file)
print("png  :", png_file)

with open("empirical_filter_set.json", "rt") as f:
    filter_list = set(json.load(f))

print("===============================")
print("EDR Tracing Heuristics")
print("IPC :", process_mode)
print("data:", data_mode)
print("===============================")

# Work on a fresh copy so the original `g` (used for searching) stays unchanged
attack_g = pickle_read(igraph_file)

print("Relabeling nodes and edges...")
relabels(attack_g)

print("Recovering vertex times...")
vertex_times(attack_g)

print("Coloring graph...")
color_graph(attack_g, seed_file, filter_list, process_mode)
attack_only(attack_g)

if data_mode in {"leaf", "all"}:
    print("Removing singleton data...")
    prune_vertices(attack_g, seed_file)

if data_mode in {"cluster", "all"} and not os.path.basename(base_fname).startswith("fs"):
    print("Applying clustering to data objects...")
    contract_clustered_files(attack_g)

print("Pruning redundant edges...")
prune_edges(attack_g)

print("Pruning orphans...")
remove_orphan_data(attack_g)

# For any path-like name, keep only final component
relabel_name(attack_g)

# Export helpers (useful for debugging / paper figures)
dump_edges(attack_g, f"{base_fname}_attack.edges")
dump_vertices(attack_g, f"{base_fname}_attack.csv")
label_file = f"{base_fname}.labels"
write_labels_file_attack_only(attack_g, base_fname, label_file)

## Step 4: Visualize the Attack Graph

This code block continues the main [reapr.py](https://bitbucket.org/sts-lab/reapr-ground-truth/src/master/utilities/reapr.py) logic to produce a visualization of the attack.

Visualizations aren't perfect using the stock iGraph layouts, regardless of whether force-directed or hierarchical algorithms are used. They're also oblivious to the vertex width. But they're good enough to get a feel for the attack. Here we're using the hierarchical Sugiyama layout.

In [ ]:

from IPython.display import Image, display

print("Plotting %s of size V=%d, E=%d..." % (pdf_file, len(attack_g.vs), len(attack_g.es)))
if attack_g.vcount() == 0:
    raise RuntimeError("Empty graph after attack_only(). Check your seeds.")

attack_g = attack_g.permute_vertices(sorted(range(attack_g.vcount()), key=lambda i: attack_g.vs["start"][i]))
layout = attack_g.layout_sugiyama()

attack_g.vs["label"] = [v["name"] for v in attack_g.vs]
attack_g.vs["shape"] = ["rectangle" for _ in attack_g.vs]
attack_g.vs["height"] = [25 for _ in attack_g.vs]
attack_g.vs["width"] = [50 + 10*(len(n)-1) for n in attack_g.vs["name"]]

attack_g.es["label"] = [e["type"] for e in attack_g.es]
attack_g.es["arrow_size"] = [1.25 for _ in attack_g.es]

igraph.plot(attack_g, pdf_file, layout=layout, bbox=(4000, 1500), margin=250)

print("Wrote:", pdf_file)

igraph.plot(attack_g, png_file, layout=layout, bbox=(4000, 1500), margin=250)
print("Wrote:", png_file)


display(Image(filename=png_file))

### Optional: Force a DAG

The Sugiyama graph layout expects a directed acyclic graph (DAG), which is part of the problem.

Using `dagify_root_forward` forces a best-effort DAG, traversing the graph from the root causes a second time and pruning edges that aren't needed for source reachability. This visualization loses a bit of the attack story but the layout is far better.

In [ ]:
from graph_helpers import dagify_root_forward

dag_g = dagify_root_forward(attack_g, root_cause_uuids)


print("attack_g: V =", attack_g.vcount(), "E =", attack_g.ecount())
print("dag_g: V =", dag_g.vcount(), "E =", dag_g.ecount())

# Plot the DAG view
dag_pdf = base_fname + "_attack_dag.pdf"
dag_png = base_fname + "_attack_dag.png"

layout = dag_g.layout_sugiyama()
igraph.plot(dag_g, dag_pdf, layout=layout, bbox=(4000, 1500), margin=250)
igraph.plot(dag_g, dag_png, layout=layout, bbox=(4000, 1500), margin=250)

print("Wrote:", dag_pdf)
print("Wrote:", dag_png)
display(Image(filename=dag_png))


## Keep Experimenting!

Does the attack graph seem complete? Are you able to identify the different steps from the attack log? Depending on which IoC's you used, further experimentation may be required.

Here's a more complete list of points of interest for the two attack graphs. Everything marked as `code` is a Vertex UUID you can copy into your list of root causes and impacts above.

For H1...
*   The attack begins with downloads from filetransfer.io, seen as connections to `172.67.200.96:443` and `104.21.13.139:443` in the logs.
*   The attack connects by reverse shell over `10.195.179.70:8080`
*   The attack connects C2 server is host at `10.195.179.70:443`. Should we be counting that as a root cause, an impact, or both?
*   The attack spreads laterally to FS on IP:PORT `130.126.136.248:22`

There are *a lot* of adversary-brought tools and data artifacts we could add to our impact label set. They aren't all necessary to identify every attack process, but there's also no harm in their inclusion:

*   `c:\users\efficientgoldfinch\downloads\drop-payloads\drop-payloads.vbe`
*   `c:\users\efficientgoldfinch\downloads\drop-payloads\drop-payloads.vbe:zone.identifier`
*   `c:\users\efficientgoldfinch\appdata\roaming\microsoft\windows\recent\drop-payloads.vbe.lnk`
*   `C:\Users\efficientgoldfinch\Downloads\drop-payloads\drop-payloads.vbe`
*   `c:\users\efficientgoldfinch\appdata\roaming\transbaseodbcdriver\screenshot__.ps1`
*   `C:\Users\efficientgoldfinch\AppData\Roaming\TransbaseOdbcDriver\screenshot__.ps1`
*   `c:\users\efficientgoldfinch\appdata\roaming\transbaseodbcdriver\screenshot__.png`
*   `c:\users\efficientgoldfinch\appdata\roaming\transbaseodbcdriver\lancraddriver.ps1`
*   `C:\Users\efficientgoldfinch\AppData\Roaming\TransbaseOdbcDriver\LanCradDriver.ps1`
*   `c:\users\efficientgoldfinch\appdata\roaming\transbaseodbcdriver\admin.txt`
*   `c:\users\efficientgoldfinch\appdata\roaming\transbaseodbcdriver\rad353f7.ps1`
*   `C:\Users\efficientgoldfinch\AppData\Roaming\TransbaseOdbcDriver\rad353F7.ps1`
*   `c:\users\efficientgoldfinch\appdata\roaming\transbaseodbcdriver\smrs.exe`
*   `c:\users\efficientgoldfinch\appdata\roaming\transbaseodbcdriver\rad353f7.ps1`
*   `C:\Users\efficientgoldfinch\AppData\Roaming\TransbaseOdbcDriver\rad353F7.ps1`
*   `c:\users\efficientgoldfinch\appdata\roaming\transbaseodbcdriver\mgscoxpsnk.txt`

For FS...

*   Lateral movement from H1 to FS happens on IP:PORT `10.195.75.103:59224`
*   Lateral movement from FS to the DC happens on IP:PORT `130.126.136.46:445`
*   The attacker uses `nslookup` to resolve the DC domain, ironically triggering a DNS request to the DC on IP:PORT `130.126.136.46:53`
*   Adversary-brought tools on the FS include `/tmp/psexec.py`

In [ ]:
# "Verbose" seeds for the H1 device. Copy and paste into Step 3.
# Notice that the same processes are implicated with the more parsimonious
# label set, but now it's much easier to see what they were up to.

root_cause_uuids = [
    # "uuid-here",
    "10.195.179.70:8080",
    "10.195.179.70:443",
    "172.67.200.96:443",
    "104.21.13.139:443"
]

impact_uuids = [
    # "uuid-here",
    "130.126.136.248:22",
    "10.195.179.70:8080",
    "10.195.179.70:443",
    r"c:\users\efficientgoldfinch\downloads\drop-payloads\drop-payloads.vbe",
    r"c:\users\efficientgoldfinch\downloads\drop-payloads\drop-payloads.vbe:zone.identifier",
    r"c:\users\efficientgoldfinch\appdata\roaming\microsoft\windows\recent\drop-payloads.vbe.lnk",
    r"C:\Users\efficientgoldfinch\Downloads\drop-payloads\drop-payloads.vbe",
    r"c:\users\efficientgoldfinch\appdata\roaming\transbaseodbcdriver\screenshot__.ps1",
    r"C:\Users\efficientgoldfinch\AppData\Roaming\TransbaseOdbcDriver\screenshot__.ps1",
    r"c:\users\efficientgoldfinch\appdata\roaming\transbaseodbcdriver\screenshot__.png",
    r"c:\users\efficientgoldfinch\appdata\roaming\transbaseodbcdriver\lancraddriver.ps1",
    r"C:\Users\efficientgoldfinch\AppData\Roaming\TransbaseOdbcDriver\LanCradDriver.ps1",
    r"c:\users\efficientgoldfinch\appdata\roaming\transbaseodbcdriver\admin.txt",
    r"c:\users\efficientgoldfinch\appdata\roaming\transbaseodbcdriver\rad353f7.ps1",
    r"C:\Users\efficientgoldfinch\AppData\Roaming\TransbaseOdbcDriver\rad353F7.ps1",
    r"c:\users\efficientgoldfinch\appdata\roaming\transbaseodbcdriver\smrs.exe",
    r"c:\users\efficientgoldfinch\appdata\roaming\transbaseodbcdriver\rad353f7.ps1",
    r"C:\Users\efficientgoldfinch\AppData\Roaming\TransbaseOdbcDriver\rad353F7.ps1",
    r"c:\users\efficientgoldfinch\appdata\roaming\transbaseodbcdriver\mgscoxpsnk.txt"
    ]


# Semi 'verbose' seeds for FS.

root_cause_uuids = [
    "10.195.75.103:59224"
]

impact_uuids = [
    "130.126.136.46:445",
    "130.126.136.46:53",
    "/tmp/psexec.py"
]